# 05. Networks and Space: Visualizing and Analyzing Spatial Networks

This notebook focuses on **spatially embedded networks** — graphs whose nodes live at *real* locations and whose edges are constrained, at least in part, by physical geography. The central question is not just *which* nodes connect to which others, but *how* the graph structure interacts with geographic position.

It follows the static abstract-network notebooks and shifts the emphasis from abstract layout design to **spatial fidelity** and **spatial analysis**.

## What you will build, step by step

We will progress from conceptual grounding, through increasingly faithful visual representations, to analytical views that only make sense once space is respected:

1. **What makes a network spatial?** A short theory primer and a warm-up using a US inter-city graph that already sits in `data/major_us_cities.json`. We compare an abstract spring layout against geographic coordinates on the same graph — the cost of ignoring space becomes impossible to miss.
2. **Loading and inspecting a road network** — the walkable street network of Moncalieri, Italy, downloaded with OSMnx and cached on disk. We look at one node and one edge so that every later plotting decision is grounded in the actual data structure.
3. **From topology to geography** — a weak force-directed view, then geographic node coordinates, then full street geometry in a GeoDataFrame, then a projected local-metric CRS. Four representations, each a strict improvement in *spatial fidelity*.
4. **Analytical spatial views** — edge-length distribution, node degree in space, shortest path vs. great-circle detour, and edge betweenness mapped back onto the street network. These are the plots that only a spatial representation can support.
5. **Geographic context** — a basemap overlay with `contextily`, so the network lives in a place readers can recognize.
6. **A print-ready spatial figure** — a polished synthesis figure, with the shortest path overlaid, a compact summary card, a scale bar and a north arrow.
7. **Exercises and takeaways** — three exercise prompts of increasing ambition, and a set of lessons that bridge back to the lecture.

## Learning goals

By the end of the notebook you should be able to:

- define a *spatial network* and name three or four properties that distinguish spatial graphs from purely topological ones;
- distinguish abstract graph layouts from spatially meaningful coordinates, and argue *when* each is appropriate;
- inspect road-network nodes and edges as geographic data with OSMnx and GeoPandas;
- move a graph from long/lat coordinates to a projected CRS and explain why that matters for metric reasoning;
- compute and visualize basic spatial-network quantities — edge-length distributions, node degree in place, shortest paths, detour/circuity ratios, and edge betweenness;
- overlay a street network on a basemap for geographic context and read the result critically;
- end with a print-ready spatial-network figure that carries a scale bar and a north arrow, and that you can defend as a design decision.

In [ ]:
from pathlib import Path
import sys
import json
import warnings

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import FancyArrowPatch
import networkx as nx
from networkx.readwrite import json_graph
import numpy as np
import osmnx as ox
from sklearn.preprocessing import MinMaxScaler

try:
    import contextily as ctx
    HAS_CTX = True
except Exception:
    ctx = None
    HAS_CTX = False

cwd = Path.cwd().resolve()
tutorials_dir = cwd.parent if cwd.name == "04-networks" else cwd / "tutorials"
if str(tutorials_dir) not in sys.path:
    sys.path.insert(0, str(tutorials_dir))

from dataviz_utils import *

set_seeds()
TEXT = apply_teaching_rc(grid=False, font_base=11.5)

# High-resolution inline rendering. Retina doubles the raster on HiDPI screens;
# the DPI bumps keep savefig / nbconvert exports crisp for printing.
try:
    from IPython import get_ipython
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
except Exception:
    pass
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["savefig.bbox"] = "tight"

FIG = make_figure_size_scale(
    focus=(7.6, 7.6),
    standard=(8.6, 5.6),
    wide=(12.0, 7.0),
)

NOTEBOOK_DIR = tutorials_dir / "04-networks"
DATA_DIR = NOTEBOOK_DIR / "data"

# Silence noisy warnings from OSMnx / shapely in teaching runs.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")


def reserve_header(fig, top=0.80, bottom=0.10, left=0.07, right=0.95):
    """Reserve figure-top padding so `style_network_panel` can place titles
    and subtitles above the axes without overlapping the plot."""
    fig.subplots_adjust(top=top, bottom=bottom, left=left, right=right)
    return fig


def text_below_axes(ax, text, *, x=0.5, y=-0.04, ha="center", va="top",
                    mono=True, box=True):
    """Place a text block OUTSIDE the axes (below). y<0 is axes-fraction space
    below the axes; the enclosing `reserve_header(bottom=...)` must leave
    enough margin."""
    bbox = (dict(boxstyle="round,pad=0.30", fc="white", ec="#D5DDE7",
                 lw=0.8, alpha=0.96) if box else None)
    ax.text(
        x, y, text,
        transform=ax.transAxes, ha=ha, va=va,
        fontsize=TEXT["annotation"],
        family="monospace" if mono else None,
        bbox=bbox, clip_on=False, zorder=20,
    )


In [ ]:
# Shared drawing constants used throughout the notebook.
CITY_NODE_CMAP       = plt.cm.Blues
SOCIAL_EDGE_COLOR    = "#C8CED8"
HIGHLIGHT_EDGE_COLOR = DV_PALETTE["orange"]
STREET_EDGE_COLOR    = "#222222"
CITY_NODE_EDGE       = "#314252"
LABEL_BBOX = dict(boxstyle="round,pad=0.22", fc="white", ec="#D5DDE7",
                  lw=0.8, alpha=0.97)


def style_network_panel(ax, title=None, subtitle=None):
    """Place title + subtitle in a reserved strip above `ax`, and shrink the
    axes downwards so the text never overlaps the plot."""
    from textwrap import fill

    fig = ax.figure
    pos = ax.get_position()
    fig_width, fig_height = fig.get_size_inches()

    def pts_to_fig_y(points):
        return (points / 72.0) / fig_height

    def width_to_chars(width_fraction, chars_per_inch=11):
        panel_width_in = max(fig_width * width_fraction, 2.0)
        return max(28, int(panel_width_in * chars_per_inch))

    wrapped_title = fill(title, width=width_to_chars(pos.width, chars_per_inch=9)) if title else None
    wrapped_subtitle = fill(subtitle, width=width_to_chars(pos.width)) if subtitle else None

    title_lines = wrapped_title.count("\n") + 1 if wrapped_title else 0
    subtitle_lines = wrapped_subtitle.count("\n") + 1 if wrapped_subtitle else 0

    title_line_pts = TEXT["panel_title"] * 1.08
    subtitle_line_pts = TEXT["annotation"] * 1.22
    top_margin_pts = 2
    gap_pts = 3 if wrapped_title and wrapped_subtitle else 0
    bottom_margin_pts = 4 if (wrapped_title or wrapped_subtitle) else 0

    header_pts = top_margin_pts + bottom_margin_pts
    if wrapped_title:
        header_pts += title_lines * title_line_pts
    if wrapped_subtitle:
        header_pts += gap_pts + subtitle_lines * subtitle_line_pts

    min_header_pts = 18 if (wrapped_title or wrapped_subtitle) else 0
    max_header_fraction = 0.18
    header_height = min(
        pts_to_fig_y(max(header_pts, min_header_pts)),
        pos.height * max_header_fraction,
    )
    min_plot_fraction = 0.68
    new_height = max(pos.height - header_height, pos.height * min_plot_fraction)
    ax.set_position([pos.x0, pos.y0, pos.width, new_height])

    header_top_y = pos.y0 + pos.height - pts_to_fig_y(top_margin_pts)
    current_top = header_top_y

    if wrapped_title:
        fig.text(
            pos.x0, current_top, wrapped_title,
            ha="left", va="top",
            fontsize=TEXT["panel_title"], color="#17212B",
        )
        current_top -= pts_to_fig_y(title_lines * title_line_pts + gap_pts)

    if wrapped_subtitle:
        fig.text(
            pos.x0, current_top, wrapped_subtitle,
            ha="left", va="top",
            fontsize=TEXT["annotation"], color=DV_PALETTE["gray"],
        )

    ax.set_axis_off()
    return ax


def scale(values, minv=50, maxv=200):
    values = np.asarray(list(values), dtype=float)
    if np.allclose(values, values[0]):
        return np.full(values.shape, (minv + maxv) / 2)
    scaler = MinMaxScaler(feature_range=(minv, maxv))
    return scaler.fit_transform(values.reshape(-1, 1)).flatten()


def set_network_limits(ax, pos, pad=0.08):
    coords = np.asarray(list(pos.values()), dtype=float)
    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)
    span = np.maximum(maxs - mins, 1e-6)
    ax.set_xlim(mins[0] - span[0] * pad, maxs[0] + span[0] * pad)
    ax.set_ylim(mins[1] - span[1] * pad, maxs[1] + span[1] * pad)
    return ax


def annotate_network_nodes(
    ax, pos, nodes, *,
    labels=None, placement_mode="external_callout",
    offset_points=14, fontsize=None, fontweight="semibold",
    box_kwargs=None, connector_kwargs=None,
    max_labels=None, avoid_overlap=None, min_vertical_gap=15,
):
    """Place external-callout labels for selected nodes, pushed radially
    outward from the graph centroid with simple overlap avoidance."""
    nodes = list(nodes)
    if max_labels is not None:
        nodes = nodes[:max_labels]

    if labels is None:
        labels = {node: node for node in nodes}
    if fontsize is None:
        fontsize = TEXT["direct_label"]
    if avoid_overlap is None:
        avoid_overlap = placement_mode != "dense_internal"

    coords = np.asarray([pos[node] for node in nodes], dtype=float)
    centroid = coords.mean(axis=0)

    default_box = LABEL_BBOX if box_kwargs is None else box_kwargs
    default_connector = (
        dict(arrowstyle="-", color="#94A3B8", lw=0.9,
             shrinkA=4, shrinkB=4, alpha=0.85)
        if connector_kwargs is None else connector_kwargs
    )

    specs = []
    for node in nodes:
        x, y = pos[node]
        dx, dy = np.array([x, y]) - centroid
        if np.isclose(dx, 0) and np.isclose(dy, 0):
            dx, dy = 0.0, 1.0
        norm = max((dx**2 + dy**2) ** 0.5, 1e-6)
        specs.append({"node": node, "x": x, "y": y,
                      "dx": dx, "dy": dy, "norm": norm})

    if placement_mode == "dense_internal":
        for spec in specs:
            ox = 0.38 * offset_points * spec["dx"] / spec["norm"]
            oy = 0.38 * offset_points * spec["dy"] / spec["norm"]
            ax.annotate(
                labels[spec["node"]], xy=(spec["x"], spec["y"]),
                xytext=(ox, oy), textcoords="offset points",
                ha="left" if ox >= 0 else "right", va="center",
                fontsize=fontsize, fontweight=fontweight, color="#27313B",
                annotation_clip=False, zorder=8,
            )
        return ax

    for spec in specs:
        spec["side"] = 1 if spec["dx"] >= 0 else -1
        spec["ox"] = spec["side"] * (offset_points + 10)
        spec["oy"] = (0.78 * offset_points * spec["dy"] / spec["norm"]
                     + (7 if spec["dy"] >= 0 else -7))

    if avoid_overlap:
        for side in (-1, 1):
            group = [spec for spec in specs if spec["side"] == side]
            if not group:
                continue
            extra_push = 4 * max(len(group) - 2, 0)
            for spec in group:
                spec["ox"] = side * (offset_points + 10 + extra_push)
            group.sort(key=lambda spec: spec["oy"])
            desired = [spec["oy"] for spec in group]
            adjusted, prev = [], None
            for oy in desired:
                current = oy if prev is None else max(oy, prev + min_vertical_gap)
                adjusted.append(current); prev = current
            shift = np.mean(desired) - np.mean(adjusted)
            adjusted = [v + shift for v in adjusted]
            prev = None
            for spec, oy in zip(group, adjusted):
                current = oy if prev is None else max(oy, prev + min_vertical_gap)
                spec["oy"] = current; prev = current

    for spec in specs:
        ax.annotate(
            labels[spec["node"]], xy=(spec["x"], spec["y"]),
            xytext=(spec["ox"], spec["oy"]), textcoords="offset points",
            ha="left" if spec["ox"] >= 0 else "right",
            va="bottom" if spec["oy"] >= 0 else "top",
            fontsize=fontsize, fontweight=fontweight, color="#1F2933",
            bbox=default_box, arrowprops=default_connector,
            annotation_clip=False, zorder=10,
        )
    return ax


## 1. What makes a network spatial?

A **spatial network** is a graph in which nodes are assigned positions in some metric space — typically the 2-D surface of the Earth — and in which the structure of the edges interacts with that space. A road network is a spatial network. So is a power grid, a neuronal connectome, a cargo shipping network, a subway map, and the Internet's autonomous-system topology drawn on a globe. A Facebook friendship graph is **not** a spatial network *per se*: users have locations, but the edges do not have to cross any physical space to exist.

The moment positions matter, several things change at once:

- every edge has a **length**, and the length typically biases *which edges can exist* and *how often they are traversed*;
- the graph almost always has a natural **two-dimensional embedding**, which means we can actually *draw* it on a page with coordinates that a geographer would recognise;
- many standard graph algorithms — shortest paths, centrality, clustering — acquire a new interpretation once *distance in space* is distinguishable from *distance in the graph*.

Before looking at the data, it is worth pausing on what we mean by "spatial". A graph becomes spatial the moment we can answer three questions:

1. *Where is each node?* (A pair of coordinates per node.)
2. *What does an edge mean in space?* (A bridge, a flight, a cable, a friendship formed during college…)
3. *Can we measure a distance along the edge?* (Euclidean, great-circle, routed along a path.)

### What distinguishes spatial networks from other networks

Four properties usually come up — each has immediate consequences for how we visualise and analyse these graphs:

1. **Distance is a cost.** Building or traversing a long edge is usually more expensive than a short one, so spatial networks tend to be dominated by *local* connections. The degree distribution is almost always *bounded* (no infinite-tailed hubs, unlike many online social graphs) — a single road intersection cannot plausibly have 1 000 streets meeting at it.
2. **They are close to planar.** Many spatial networks (street grids, power grids, river networks) avoid edge crossings because physical crossings would require bridges, tunnels, or flyovers. Planarity has deep implications: by Euler's formula, the number of faces, edges, and nodes are related and the network cannot be arbitrarily dense.
3. **Clustering is strong, small-world behaviour is weakened.** Neighbours of neighbours tend to be nearby, so triangles are common. But because long-distance edges are rare, the average shortest-path length grows more like √N (2-D geometry) than like log N (random graphs). Pure small-world phenomena require a few non-local shortcuts, which many transport networks simply lack.
4. **Meaningful positions exist before we plot anything.** Unlike abstract graphs where a layout is a design choice, a spatial network comes with *real* coordinates. Drawing them anywhere else is a representation we must justify.

Classic examples that recur throughout the course: **street and road networks** (our main Moncalieri example), **airline and flight networks** (airports linked by direct flights), **power grids**, **internet backbone and submarine-cable networks**, **neural networks in the brain**, and **river drainage networks**.

### Warm-up: a US inter-city network

Before we touch street networks, let us build intuition on a small, friendly example. The file `data/major_us_cities.json` stores 50 major US cities as nodes and inter-city links (roughly the shortest-distance neighbours) as edges. Each node records a geographic location and population; each edge records a great-circle distance as weight. We load it, inspect one node so students know the attribute names, and then use it to illustrate the core point of this section.

In [ ]:
with open(DATA_DIR / "major_us_cities.json") as f:
    cities_payload = json.load(f)
G_cities = json_graph.node_link_graph(cities_payload, edges="links")
next(iter(G_cities.nodes(data=True)))


Every node already carries a `location` `[longitude, latitude]` pair and a `population`. Edges carry a `weight` (great-circle distance in km). So the graph is not just topological: it is **pre-embedded** — geography is part of the data.

In [ ]:
city_geo_pos = {node: tuple(attr["location"]) for node, attr in G_cities.nodes(data=True)}
city_populations = np.array([G_cities.nodes[n]["population"] for n in G_cities.nodes()])
city_node_sizes = scale(city_populations, minv=60, maxv=520)

# Colour cities by longitude so the east/west gradient is visible in both panels.
city_longitudes = np.array([G_cities.nodes[n]["location"][0] for n in G_cities.nodes()])
lon_norm = Normalize(vmin=city_longitudes.min(), vmax=city_longitudes.max())
city_node_colors = [lon_norm(lon) for lon in city_longitudes]

labeled_cities = ["New York NY", "Chicago IL", "Los Angeles CA", "Houston TX", "Miami FL", "Seattle WA"]
labeled_cities = [c for c in labeled_cities if c in G_cities.nodes]
print(f"{G_cities.number_of_nodes()} cities, {G_cities.number_of_edges()} inter-city edges.")
print("Node attributes include:", list(next(iter(G_cities.nodes(data=True)))[1].keys()))


### Spring layout vs geographic coordinates, side by side

We now draw the same graph twice:

- **left panel** — an abstract `spring_layout` that knows nothing about longitude or latitude. It places nodes by optimising a physical simulation of springs, so tightly-connected cities end up close on the page;
- **right panel** — the geographic layout, using each city's `(longitude, latitude)` directly as its position.

Node size encodes population and node colour encodes longitude (west ⇒ pale, east ⇒ dark blue). Edges come from the original dataset (short inter-city links). The comparison is deliberately unfair: the spring layout cannot represent space, even in principle. That is the pedagogical point.

In [ ]:
labeled_cities_dict = {c: c for c in labeled_cities}

fig, axes = plt.subplots(1, 2, figsize=FIG["wide"])
reserve_header(fig, top=0.82, bottom=0.08, left=0.05, right=0.97)

# --- Left panel: abstract spring layout (no spatial constraint) ----------
city_spring_pos = nx.spring_layout(G_cities, seed=RANDOM_SEED, weight=None)
ax = axes[0]
nx.draw_networkx_edges(
    G_cities, city_spring_pos,
    edge_color=SOCIAL_EDGE_COLOR, width=0.7, alpha=0.9, ax=ax,
)
nx.draw_networkx_nodes(
    G_cities, city_spring_pos,
    node_size=city_node_sizes, node_color=city_node_colors,
    cmap=CITY_NODE_CMAP, edgecolors=CITY_NODE_EDGE, linewidths=0.8, ax=ax,
)
annotate_network_nodes(
    ax, city_spring_pos, labeled_cities,
    labels=labeled_cities_dict,
    placement_mode="external_callout",
    offset_points=26, min_vertical_gap=18,
)
set_network_limits(ax, city_spring_pos, pad=0.32)
style_network_panel(
    ax,
    "Spring layout (no geography)",
    "Clusters reflect adjacency, not location. Coastlines disappear.",
)

# --- Right panel: geographic layout (longitude, latitude) -----------------
ax = axes[1]
nx.draw_networkx_edges(
    G_cities, city_geo_pos,
    edge_color=SOCIAL_EDGE_COLOR, width=0.7, alpha=0.9, ax=ax,
)
nx.draw_networkx_nodes(
    G_cities, city_geo_pos,
    node_size=city_node_sizes, node_color=city_node_colors,
    cmap=CITY_NODE_CMAP, edgecolors=CITY_NODE_EDGE, linewidths=0.8, ax=ax,
)
annotate_network_nodes(
    ax, city_geo_pos, labeled_cities,
    labels=labeled_cities_dict,
    placement_mode="external_callout",
    offset_points=26, min_vertical_gap=18,
)
set_network_limits(ax, city_geo_pos, pad=0.22)
style_network_panel(
    ax,
    "Geographic coordinates (lon, lat)",
    "The shape of the continental US is recognisable; the graph now carries distance.",
)

plt.show()


With the warm-up done, we now move to a more demanding example: the *walkable street network of Moncalieri, Italy*. Streets are the canonical spatial network — dense, constrained by physics, and interesting at every scale. The remainder of the notebook is built around this single dataset so that every design and analysis step can be compared on identical ground.

## 2. Loading and inspecting a road network

We now load a *real* street network and look at it one layer at a time. The dataset is the walkable street network of Moncalieri, Italy, downloaded from OpenStreetMap through **OSMnx** and cached on disk as `data/moncalieri_walk.graphml` so this notebook runs offline after the first download.

The first cells load the graph, inspect one node, and inspect one edge so that the plotting decisions that follow are grounded in the actual data structure rather than abstractions. This is useful because street networks already have meaningful positions in space: they ship pre-embedded.

In [ ]:
# Specify the name of the city
city = "Moncalieri, IT"
graph_path = DATA_DIR / "moncalieri_walk.graphml"

if graph_path.exists():
    C = ox.load_graphml(graph_path)
else:
    try:
        C = ox.graph_from_place(city, network_type="walk")
    except Exception as exc:
        raise RuntimeError(
            "The street-network example needs an internet connection the first time it runs. "
            "Once downloaded, the GraphML file is cached locally."
        ) from exc
    ox.save_graphml(C, graph_path)

G_nx = nx.relabel.convert_node_labels_to_integers(C)
print(f"{G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")


### Examining the Node Data

The next cell inspects one node record so students can see which attributes are available at an intersection. This is a useful step because plotting decisions later in the notebook depend on knowing what the graph actually stores.


In [ ]:
next(iter(C.nodes(data=True)))

### Examining the Edge Data

The same inspection step is useful for edges. Street-network edges often carry geometric and length information that matters much more than in abstract social-network examples.


In [ ]:
next(iter(C.edges(data=True)))

## 3. From topology to geography: four representations of the same network

For abstract social networks, a force-directed layout can reveal structure. For road networks, the same layout can destroy the very geometry the reader needs to see.

The next sequence moves deliberately from a **weak, non-spatial** representation to increasingly faithful **spatial** ones. There are four stages, and each is a strict improvement over the one before it:

1. **Spring layout** — ignores space entirely, treats the graph as pure topology.
2. **Geographic node coordinates** — uses the `(x, y)` attributes already stored on the graph; edges remain straight lines.
3. **Full street geometry from GeoDataFrames** — edges acquire their real, curved shapes.
4. **Projected local-metric CRS** — coordinates live in metres, not degrees, so lengths and distances become trustworthy.

Reading §§3–5 in this order lets students see exactly *what* each step adds and *why*.

### Function Guide: `nx.draw_networkx_nodes` and `nx.draw_networkx_edges`

In road networks we often draw nodes and edges separately because they play very different visual roles.

**Why use separate functions**
- road segments usually form the main geometry
- intersections may need to be tiny, muted, or even omitted
- the same `pos` dictionary can be reused for both layers

**Parameters to notice**
- `pos`: the coordinate dictionary
- `node_size`, `node_color`, `edgecolors`: for intersections
- `width`, `edge_color`, `alpha`, `arrows`: for street segments


In [ ]:
spring_pos = nx.spring_layout(G_nx, iterations=100, weight="length", seed=RANDOM_SEED)

The next plot uses the weak comparison on purpose. Read it as evidence of what is lost when topology is shown without preserving space.

In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig)
nx.draw_networkx_nodes(G_nx, spring_pos, node_size=5, node_color="white", edgecolors="black", linewidths=0.4, ax=ax)
nx.draw_networkx_edges(G_nx, spring_pos, width=0.2, edge_color=STREET_EDGE_COLOR, arrows=False, alpha=0.7, ax=ax)
style_network_panel(ax, "Abstract spring layout", "Topology only — no geography. A useful weak baseline.")
plt.show()


### Using Geographic Coordinates

The next two cells move from abstract layout to spatial layout. The code first extracts longitude and latitude from the node attributes, then redraws the graph using those coordinates.

This is a simple but important step: once the positions come from the data, the plot can support geographic interpretation.


In [ ]:
geo_pos_street = {
    node: (attributes["x"], attributes["y"])
    for node, attributes in G_nx.nodes(data=True)
}


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig)
nx.draw_networkx_nodes(G_nx, geo_pos_street, node_size=5, node_color="white", edgecolors="black", linewidths=0.4, ax=ax)
nx.draw_networkx_edges(G_nx, geo_pos_street, width=0.2, edge_color=STREET_EDGE_COLOR, arrows=False, alpha=0.7, ax=ax)
style_network_panel(ax, "Geographic node coordinates", "Positions from the data; edges still drawn as straight segments.")
plt.show()


The previous method still treats each street segment as a straight line between its endpoints, even when the real street geometry contains curves or intermediate vertices.


## 4. Full street geometry with GeoDataFrames

Node coordinates improve the layout, but they still draw each street as a straight segment between endpoints. Real street networks often include curved geometries and intermediate vertices — think of a winding via in the old town, or a highway that gently bends around a hill.

GeoDataFrames preserve that geometry, so this is the first representation in the notebook that is faithful to the full street shape. It is also the representation that opens the door to the full geospatial Python stack (GeoPandas, Shapely, PyProj, contextily), which we will exploit from here on.

Another best practice that we touch on in the next section is to **project** the graph before metric operations such as measuring lengths, buffering, or comparing shapes in local planar space.

### Function Guide: `ox.graph_to_gdfs`

`ox.graph_to_gdfs` converts an OSMnx graph into GeoDataFrames. This matters because GeoDataFrames preserve real street geometry, not just endpoint coordinates.

**What it returns**
- a node GeoDataFrame
- an edge GeoDataFrame

Once the data is in GeoDataFrames, it can be plotted with GeoPandas and handled with the broader geospatial stack.


In [ ]:
street_nodes_gdf, street_edges_gdf = ox.graph_to_gdfs(C)

Now the graph is no longer a straight-line node-link approximation. The plotted edge layer carries the real street geometry.

In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig)
street_edges_gdf.plot(ax=ax, linewidth=0.5, edgecolor=STREET_EDGE_COLOR)
street_nodes_gdf.plot(ax=ax, markersize=5, facecolor="white", edgecolor="black", linewidth=0.5)
style_network_panel(ax, "Full street geometry from GeoDataFrames", "Real curved street shapes, not straight segments.")
plt.show()


## 5. Projecting the street network into a local metric CRS

Projection is a core geospatial best practice. Unprojected coordinates (longitude, latitude in degrees) are fine for web maps and geographic display, but projected coordinates are the safer basis for *local metric reasoning* because distances and geometry are expressed in a planar CRS — typically in **metres**.

Concretely, here is what changes:

- A degree of longitude is not a constant distance: near the equator it is roughly 111 km, but at 60° latitude it shrinks to 55 km. If we measured edge lengths in degrees we would systematically underestimate horizontal distances in Europe compared to the equator.
- Angles and shapes distort less in a local projection (OSMnx picks an appropriate UTM zone automatically), which is the right behaviour when the network is small enough to fit comfortably inside one zone.

Measuring street length, computing betweenness weighted by distance, buffering around intersections, comparing neighbourhoods by size — all of these require metres, not degrees.

### Function Guide: `ox.project_graph`

`ox.project_graph` reprojects the network into a local planar CRS. This is the safer basis for metric reasoning because:
- distances and geometry are expressed in projected units
- local shapes are more faithful for measurement and comparison

The next two cells keep projection and plotting separate so students can see exactly when the CRS changes.


In [ ]:
C_proj = ox.project_graph(C)
street_nodes_proj_gdf, street_edges_proj_gdf = ox.graph_to_gdfs(C_proj)

Before plotting, it is informative to check *which* CRS the graph was moved into. OSMnx picks a local UTM zone automatically. We can read the `crs` attribute of the edges GeoDataFrame.

In [ ]:
print("Unprojected CRS:", street_edges_gdf.crs)
print("Projected CRS  :", street_edges_proj_gdf.crs)
print()
print("Sample length in degrees (unprojected):", round(street_edges_gdf.geometry.iloc[0].length, 6))
print("Sample length in metres  (projected)  :", round(street_edges_proj_gdf.geometry.iloc[0].length, 2))


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig)
street_edges_proj_gdf.plot(ax=ax, linewidth=0.5, edgecolor=STREET_EDGE_COLOR)
style_network_panel(ax, "Projected street geometry", "In a local UTM metric CRS — lengths and distances now live in metres.")
plt.show()


### Using OSMnx's Built-in Plotting Functions

OSMnx's plotting helpers are useful because they encode a lot of street-network-specific logic already. This is a good example of a broader best practice: once the data type becomes specialized, use tools that understand the data structure rather than rebuilding every plot from primitives.


In [ ]:
ox.plot_graph(
    C,
    bgcolor="white",
    edge_color="black",
    edge_alpha=1.0,
    node_color="white",
    node_edgecolor="black",
    node_size=5,
)


## 6. Analytical spatial views

So far the notebook has been entirely about **representation** — four increasingly faithful ways of *drawing* the same street network. Representation is the foundation, but the whole reason to preserve spatial fidelity is that it unlocks **analysis**.

Here we build four short analytical views that only make sense once the network is spatial:

- **6a. Edge-length distribution** — how long is a typical street segment?
- **6b. Node degree in space** — where are dead-ends, T-junctions, crossroads?
- **6c. Shortest path and detour ratio** — how much longer is the walked route than the crow-fly distance?
- **6d. Edge betweenness, mapped back onto space** — which edges carry the most shortest-path traffic, and where do they live geographically?

Each view reuses the GeoDataFrames we already built, so the computation stays light.

### 6a. How long is a street? The edge-length distribution

In a street network, the edge length is *literally* how far you walk across that block. The distribution is almost always **heavy-tailed on a log axis**: there are many short residential segments, fewer medium segments, and a long tail of rare, long connectors (bypasses, bridges, avenues).

Knowing the median segment length is a hygiene check for everything that follows:

- it sets a sensible scale for buffer operations and spatial joins;
- it gives a back-of-the-envelope intuition for how many edges a typical shortest path traverses;
- it is usually the single most honest number to put into a headline summary.

In [ ]:
edge_lengths_m = street_edges_proj_gdf.geometry.length.values
length_median_m = float(np.median(edge_lengths_m))
length_mean_m = float(np.mean(edge_lengths_m))
total_length_km = float(edge_lengths_m.sum() / 1000)

fig, ax = plt.subplots(figsize=FIG["standard"])
reserve_header(fig, top=0.82, bottom=0.14, left=0.10, right=0.97)
ax.hist(
    edge_lengths_m,
    bins=np.logspace(np.log10(max(edge_lengths_m.min(), 1)), np.log10(edge_lengths_m.max()), 40),
    color=DV_PALETTE["blue"], alpha=0.85, edgecolor="white", linewidth=0.4,
)
ax.set_xscale("log")
ax.axvline(length_median_m, color=DV_PALETTE["orange"], linestyle="--", linewidth=1.2,
           label=f"median = {length_median_m:.0f} m")
ax.axvline(length_mean_m, color="#111111", linestyle=":", linewidth=1.2,
           label=f"mean = {length_mean_m:.0f} m")
ax.set_xlabel("street segment length (m, log scale)")
ax.set_ylabel("number of segments")
ax.legend(loc="upper right", frameon=True, framealpha=0.95, fontsize=TEXT["annotation"])
style_axis(ax, grid="y")

# Title above the axes (axes-style plot, not a panel — keep set_title but give
# it clear padding so it does not collide with the top bars).
ax.set_title(
    f"Edge-length distribution  ·  total walkable network: {total_length_km:,.1f} km",
    fontsize=TEXT["panel_title"], color="#17212B", loc="left", pad=12,
)
plt.show()


### 6b. Node degree in space

In a street network, the `street_count` attribute records how many streets meet at an intersection. Degree-1 nodes are **dead-ends** (cul-de-sacs), degree-2 nodes are simple **through-points** (often artefacts of where the dataset decided to split a street into segments), degree-3 nodes are **T-junctions**, and degree-≥4 nodes are **crossroads**.

The distribution of these degrees tells you a lot about the street morphology — a Manhattan-style grid is dominated by 4-way crossroads, an organic old town is dominated by 3-way T-junctions, and a newly-built suburb tends to be full of dead-ends. We encode the degree with colour and plot it on top of the projected geometry.

In [ ]:
G_nx_proj_nodes = list(C_proj.nodes())

# `street_count` may be stored as int or string depending on OSMnx version —
# normalise once and reuse in the map cell below.
street_counts = np.array([
    int(C.nodes[n].get("street_count", 0) or 0) for n in G_nx_proj_nodes
])
pct_four_way = 100 * float((street_counts >= 4).mean())
print(f"Nodes: {len(G_nx_proj_nodes):,}  |  4-way crossroads: {pct_four_way:.1f} %")


In [ ]:
degree_categories = {
    1: ("dead end (deg 1)", DV_PALETTE["orange"]),
    2: ("through-street (deg 2)", "#C9D1DA"),
    3: ("T-junction (deg 3)", DV_PALETTE["blue"]),
    4: ("crossroads (deg ≥ 4)", "#1F3B5C"),
}

# Collapse deg ≥ 4 into a single top bucket for the legend.
street_counts_capped = np.clip(street_counts, 1, 4)

fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig, top=0.82, bottom=0.06, left=0.03, right=0.97)
ax.set_facecolor("#F8FAFC")

# Base edge layer
street_edges_proj_gdf.plot(ax=ax, linewidth=0.35, color="#D9DEE4", zorder=1)

# Each degree category as its own scatter → clean legend + explicit z-ordering.
coords_arr = np.array([
    (C_proj.nodes[n]["x"], C_proj.nodes[n]["y"]) for n in G_nx_proj_nodes
])
for deg in [2, 3, 1, 4]:  # draw background bucket first, then foreground
    mask = street_counts_capped == deg
    if not mask.any():
        continue
    label, color = degree_categories[deg]
    ax.scatter(
        coords_arr[mask, 0], coords_arr[mask, 1],
        s=14 if deg in (2, 3) else 28,
        c=color, edgecolors="white", linewidths=0.4,
        label=f"{label}  ({mask.sum():,})",
        zorder=3 if deg in (1, 4) else 2,
    )

# Legend anchored BELOW the axes so it never covers data.
ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2,
    frameon=True, framealpha=0.95,
    fontsize=TEXT["annotation"], title="intersection type",
    title_fontsize=TEXT["annotation"],
)
style_network_panel(
    ax,
    "Intersections coloured by street count",
    "Dead-ends and 4-way crossroads mark the morphological signature of the district.",
)
plt.show()


### 6c. Shortest path and the detour/circuity ratio

A classic spatial-network question is: *if I have to walk from A to B along the streets, how much further do I travel than a straight line?* The ratio

$$\text{detour ratio} \;=\; \frac{\text{route length along streets}}{\text{great-circle (or Euclidean) distance from A to B}}$$

is sometimes called the **circuity** of the shortest path. It is always ≥ 1. In a perfect grid the ratio approaches 1.27 (≈ √2 on long diagonals); in a very organic old town it can climb above 1.5; river crossings can produce dramatic spikes because the street network is forced through a few bridges.

We pick two nodes near opposite corners of the bounding box, compute the length-weighted shortest path between them, overlay it on the street canvas, and print the detour ratio.

In [ ]:
# Build the undirected view + largest connected component ONCE: both the
# shortest-path cell (here) and the edge-betweenness cell (§6d) need them.
G_und = nx.Graph(C_proj)
largest_cc_nodes = max(nx.connected_components(G_und), key=len)
G_und_largest_cc = G_und.subgraph(largest_cc_nodes).copy()

# `C_proj_cc` is the directed MultiDiGraph restricted to the same CC — we use
# it for shortest_path because it still carries per-edge length attributes.
C_proj_cc = C_proj.subgraph(largest_cc_nodes).copy()


# Pick two nodes near the south-west and north-east corners of the bounding box
# so the route has to cross a meaningful chunk of the network.
nodes_list = list(C_proj_cc.nodes())
coords_proj = np.array([
    (C_proj_cc.nodes[n]["x"], C_proj_cc.nodes[n]["y"]) for n in nodes_list
])
sw_idx = int(np.argmin(coords_proj[:, 0] + coords_proj[:, 1]))
ne_idx = int(np.argmax(coords_proj[:, 0] + coords_proj[:, 1]))
orig_node = nodes_list[sw_idx]
dest_node = nodes_list[ne_idx]

# Length-weighted shortest path on the projected graph, restricted to the CC.
try:
    path_nodes = nx.shortest_path(C_proj_cc, source=orig_node, target=dest_node, weight="length")
except nx.NetworkXNoPath:
    path_nodes = [orig_node, dest_node]


# Look up each path edge (u, v, key) in the edges GDF, falling back to the
# reversed orientation (v, u, key) when the stored direction is the opposite.
def pick_path_edge_key(u, v):
    fwd = C_proj_cc.get_edge_data(u, v)
    rev = C_proj_cc.get_edge_data(v, u)
    if fwd:
        k = min(fwd, key=lambda k: fwd[k].get("length", np.inf))
        return (u, v, k)
    if rev:
        k = min(rev, key=lambda k: rev[k].get("length", np.inf))
        return (v, u, k)
    return None


path_edge_keys = [pick_path_edge_key(u, v) for u, v in zip(path_nodes[:-1], path_nodes[1:])]
path_edge_keys = [k for k in path_edge_keys if k is not None and k in street_edges_proj_gdf.index]
path_edges_gdf = street_edges_proj_gdf.loc[path_edge_keys]

# Route length (metres) = total geometry length of the traversed edges.
route_length_m = float(path_edges_gdf.geometry.length.sum())

# Crow-fly distance in metres (projected CRS is in metres).
orig_attrs = C_proj_cc.nodes[orig_node]
dest_attrs = C_proj_cc.nodes[dest_node]
crow_fly_m = float(np.hypot(dest_attrs["x"] - orig_attrs["x"],
                            dest_attrs["y"] - orig_attrs["y"]))
detour_ratio = route_length_m / crow_fly_m if crow_fly_m else float("nan")

print(f"Route length (along streets): {route_length_m / 1000:.2f} km")
print(f"Crow-fly distance           : {crow_fly_m / 1000:.2f} km")
print(f"Detour ratio (circuity)     : {detour_ratio:.2f}×")


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
# Extra bottom margin reserved for the outside stats strip.
reserve_header(fig, top=0.82, bottom=0.14, left=0.03, right=0.97)
ax.set_facecolor("#F8FAFC")

# Muted street canvas
street_edges_proj_gdf.plot(ax=ax, linewidth=0.45, color="#C9D1DA", zorder=1)

# Highlight the walked path
path_edges_gdf.plot(ax=ax, linewidth=2.4, color=HIGHLIGHT_EDGE_COLOR, zorder=3)

# Mark origin and destination
origin_xy = (orig_attrs["x"], orig_attrs["y"])
dest_xy   = (dest_attrs["x"], dest_attrs["y"])
ax.scatter(*origin_xy, s=80, color=HIGHLIGHT_EDGE_COLOR, edgecolor="white", linewidth=1.6, zorder=4)
ax.scatter(*dest_xy,   s=80, color=HIGHLIGHT_EDGE_COLOR, edgecolor="white", linewidth=1.6, zorder=4)

ax.annotate(
    "start", xy=origin_xy, xytext=(14, 10), textcoords="offset points",
    fontsize=TEXT["direct_label"], fontweight="bold", color="#1F2933",
    bbox=LABEL_BBOX,
)
ax.annotate(
    "end", xy=dest_xy, xytext=(-14, -12), textcoords="offset points",
    ha="right", va="top",
    fontsize=TEXT["direct_label"], fontweight="bold", color="#1F2933",
    bbox=LABEL_BBOX,
)

# Stats strip OUTSIDE the axes, so it never covers any street.
stats_text = (
    f"route {route_length_m / 1000:.2f} km   "
    f"·   crow-fly {crow_fly_m / 1000:.2f} km   "
    f"·   detour {detour_ratio:.2f}×"
)
text_below_axes(ax, stats_text, x=0.5, y=-0.04, ha="center", va="top")

style_network_panel(
    ax,
    "Shortest path on the real street network",
    "The walked route (orange) is longer than the crow-fly distance; the ratio is the detour.",
)
plt.show()


### 6d. Edge betweenness, mapped back onto space

**Edge betweenness** of an edge `e` counts the number of shortest paths that pass over `e`. Edges with high betweenness are the *bottlenecks*: removing them forces many routes to detour.

For a pure topological graph this is already a useful statistic. For a *spatial* graph it is much richer, because we can now plot the score **on the map**. Edges that score highest are usually the main arterials — and the plot turns a purely structural metric into a claim about the city's physical layout.

For performance, we compute betweenness on the **simple, undirected, largest-component** view of the projected graph, sampled with `k=300` source nodes. Length-weighted shortest paths are the right choice for a walk network.

In [ ]:
# Reuse the undirected largest-CC graph already built in §6c.
bc_sample = min(300, G_und_largest_cc.number_of_nodes())
edge_bc = nx.edge_betweenness_centrality(
    G_und_largest_cc, k=bc_sample, weight="length",
    seed=RANDOM_SEED, normalized=True,
)

# Map the scores back onto the projected edges GeoDataFrame.
# `edge_bc` keys are 2-tuples (u, v); the GDF is indexed by (u, v, key). Try
# both orderings because the undirected lookup ignores direction.
edges_bc_gdf = street_edges_proj_gdf.reset_index()
edges_bc_gdf["edge_bc"] = [
    edge_bc[(u, v)] if (u, v) in edge_bc else edge_bc.get((v, u), 0.0)
    for u, v in zip(edges_bc_gdf["u"], edges_bc_gdf["v"])
]
print(f"Edge-betweenness sampled on {bc_sample} sources "
      f"(of {G_und_largest_cc.number_of_nodes()} nodes).")
print(f"Max score = {edges_bc_gdf['edge_bc'].max():.4g}, "
      f"median = {edges_bc_gdf['edge_bc'].median():.4g}")


In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig, top=0.82, bottom=0.06, left=0.03, right=0.92)
ax.set_facecolor("#F8FAFC")

# Base layer: all edges muted and thin
edges_bc_gdf.plot(ax=ax, linewidth=0.25, color="#D9DEE4", zorder=1)

# Highlight only the top-quantile edges; this keeps the figure legible.
quantile_cut = 0.80
top_mask = edges_bc_gdf["edge_bc"] >= edges_bc_gdf["edge_bc"].quantile(quantile_cut)
edges_bc_top = edges_bc_gdf[top_mask].copy()
norm = Normalize(
    vmin=edges_bc_top["edge_bc"].min(),
    vmax=edges_bc_top["edge_bc"].max(),
)
edges_bc_top.plot(
    ax=ax, linewidth=1.2, column="edge_bc", cmap=CMAP_SEQUENTIAL, norm=norm, zorder=2,
)

# Thin colorbar
sm = ScalarMappable(cmap=CMAP_SEQUENTIAL, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label(f"edge betweenness (top {int((1 - quantile_cut) * 100)} %)", fontsize=TEXT["annotation"])
cbar.ax.tick_params(labelsize=TEXT["annotation"] - 1)

style_network_panel(
    ax,
    "Edge betweenness, mapped back onto the street network",
    f"Only the top {int((1 - quantile_cut) * 100)} % of edges are coloured "
    f"(k={bc_sample} sources, length-weighted). Hot edges are geographic spines.",
)
plt.show()


**Reading the plot.** Edges burning bright red/orange are the topological *spines* of the network — remove them and shortest paths have to detour significantly. In a city like Moncalieri, those spines usually coincide with the historical main streets and the river crossings. This is a good example of why **spatial + structural** analysis beats either dimension on its own: the colours come from graph topology, but the *story* only becomes readable once we draw them in geographic space.

## 7. Geographic context: a basemap overlay

A spatial-network figure can be analytically correct and still leave a reader uncertain about *where* it is. A **basemap** — a light, neutral raster of streets, water, and administrative areas — anchors the network in a place the reader can recognise. Placed correctly, it disambiguates the figure; placed wrong, it overpowers the network.

We use the `contextily` library, which downloads and caches map tiles from providers such as CartoDB, OpenStreetMap, and Stamen. A few practical rules:

1. **Match the CRS.** Tile providers serve in **Web Mercator (EPSG:3857)**. The tiles will align with your network only if you either reproject the network to Web Mercator before plotting, or tell `contextily` the CRS of your axes with the `crs=` argument.
2. **Pick a quiet style.** `CartoDB.Positron` is a light, near-monochrome map that stays out of the way. Avoid satellite tiles unless the image itself is the point.
3. **Respect attribution.** `contextily` prints the required attribution automatically; leave it in the figure.
4. **Have a graceful fallback.** Tile fetches require the network. We wrap the call in `try/except` so the notebook still runs offline.

In [ ]:
fig, ax = plt.subplots(figsize=FIG["focus"])
reserve_header(fig, top=0.82, bottom=0.06, left=0.03, right=0.97)

# Plot the projected network; contextily needs us to pass the same CRS.
street_edges_proj_gdf.plot(ax=ax, linewidth=0.55, color="#243746", zorder=2)

if HAS_CTX:
    try:
        ctx.add_basemap(
            ax,
            crs=street_edges_proj_gdf.crs,
            source=ctx.providers.CartoDB.Positron,
            zorder=1,
        )
        subtitle = "CartoDB.Positron tiles give a quiet backdrop; the network stays in the foreground."
    except Exception as exc:
        subtitle = f"(Basemap tiles unavailable — running offline: {exc.__class__.__name__}.)"
else:
    subtitle = "(contextily not installed — install it to see the basemap underneath.)"

style_network_panel(
    ax,
    "Moncalieri walk network with a basemap underlay",
    subtitle,
)
plt.show()


## 8. A print-ready spatial figure

The final figure keeps the professional goal narrow and clear:

- preserve the full street geometry;
- suppress unnecessary node symbols;
- use a projected CRS so the geometry is interpreted in consistent local metric space;
- keep the page quiet enough that the spatial form of the network can lead;
- **add one analytical layer** (our shortest path from §6c) and a compact summary card that records the most honest quantities we have computed.

For road networks, this kind of geometric fidelity plus a single, purposeful overlay is often more valuable than labelling intersections or piling on encodings. The figure below also adds a *scale bar* and a *north arrow* — the two bits of spatial metadata any printed map should carry.

### Step 1 — compute the summary quantities

Before composing the figure we refresh every number we want to show, so the plotting cell that follows is pure layout logic.

In [ ]:
# All quantities shown in the print-ready figure come from earlier sections.
# Alias for readability in the figure cell below.
median_edge_m   = length_median_m
n_intersections = len(street_nodes_proj_gdf)

print(f"total walkable streets : {total_length_km:,.1f} km")
print(f"intersections          : {n_intersections:,}")
print(f"median segment length  : {median_edge_m:.0f} m")
print(f"4-way crossroads       : {pct_four_way:.1f} %")
print(f"detour ratio (SW→NE)   : {detour_ratio:.2f}×")


### Step 2 — compose the final figure

The figure is an **infographic**, not just a map. It has three kinds of marks:

- **Data.** The muted street network, with the highlighted shortest path.
- **Spatial reference.** A scale bar in metres and a north arrow — both required any time the figure is printed or read in isolation.
- **Interpretation.** A compact stat card (the KPI-style numbers) and a title block that tells the reader what to look at first.

In [ ]:
# -----------------------------------------------------------------------------
# Colour + typography system
# -----------------------------------------------------------------------------
BG       = "#FAFBFC"
INK      = "#0F172A"    # ~slate-900
INK_SOFT = "#475569"    # ~slate-600
MUTED    = "#E2E8F0"    # ~slate-200
STREET   = "#1F2937"    # street ink
ACCENT   = HIGHLIGHT_EDGE_COLOR

TITLE_FS   = 20
SUBT_FS    = 11.5
KPI_NUM_FS = 20
KPI_LAB_FS = 9.2
MICRO_FS   = 8.8


# -----------------------------------------------------------------------------
# Figure: stat strip + map axes.
# GridSpec gives the map a generous square canvas and a slim top band for
# title + KPIs, so no text ever lands on a street.
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(10.4, 11.8), facecolor=BG)
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1.0, 5.2],
    left=0.06, right=0.94, top=0.97, bottom=0.055,
    hspace=0.02,
)

# -- Top: title + subtitle + KPI band ----------------------------------------
ax_top = fig.add_subplot(gs[0, 0])
ax_top.set_facecolor(BG)
ax_top.set_axis_off()
ax_top.set_xlim(0, 1)
ax_top.set_ylim(0, 1)

ax_top.text(
    0.0, 0.92, "Moncalieri — walkable street network",
    fontsize=TITLE_FS, fontweight="semibold", color=INK, va="top",
)
ax_top.text(
    0.0, 0.68,
    "A projected view in a local metric CRS (UTM 32N). "
    "Full street geometry in dark ink, the length-weighted shortest path "
    "from SW to NE corner in orange.",
    fontsize=SUBT_FS, color=INK_SOFT, va="top",
)

# KPI cards — four evenly-spaced numbers at the bottom of the top axes.
kpis = [
    (f"{total_length_km:,.1f}", "km of walkable streets"),
    (f"{n_intersections:,}",    "intersections"),
    (f"{int(round(median_edge_m))} m", "median segment"),
    (f"{detour_ratio:.2f}×",    "detour ratio (SW → NE)"),
]
kpi_y_num = 0.30
kpi_y_lab = 0.07
for i, (num, lab) in enumerate(kpis):
    x = 0.015 + i * 0.2475
    ax_top.text(
        x, kpi_y_num, num,
        fontsize=KPI_NUM_FS, fontweight="semibold", color=INK, va="center",
    )
    ax_top.text(
        x, kpi_y_lab, lab,
        fontsize=KPI_LAB_FS, color=INK_SOFT, va="center",
    )
ax_top.plot(
    [0.0, 1.0], [-0.04, -0.04],
    color=MUTED, lw=0.9, clip_on=False,
)

# -- Map -----------------------------------------------------------------------
ax = fig.add_subplot(gs[1, 0])
ax.set_facecolor(BG)
ax.set_aspect("equal", adjustable="datalim")

# Three-pass edge rendering: soft base, glow under path, sharp path on top.
street_edges_proj_gdf.plot(ax=ax, linewidth=0.45, color=STREET, alpha=0.78, zorder=1)
path_edges_gdf.plot(ax=ax, linewidth=5.5, color=ACCENT, alpha=0.16, zorder=2)
path_edges_gdf.plot(ax=ax, linewidth=2.6, color=ACCENT, alpha=0.98, zorder=3)

# Start / end markers with small callout chips.
origin_xy = (orig_attrs["x"], orig_attrs["y"])
dest_xy   = (dest_attrs["x"], dest_attrs["y"])
for pt, txt, dx, dy, ha in [
    (origin_xy, "START (SW)", 16,  18, "left"),
    (dest_xy,   "END (NE)",  -16, -14, "right"),
]:
    ax.scatter(*pt, s=130, facecolor="white", edgecolor=ACCENT, linewidth=2.2, zorder=4)
    ax.scatter(*pt, s=34, facecolor=ACCENT, edgecolor="white", linewidth=1.0, zorder=5)
    ax.annotate(
        txt, xy=pt, xytext=(dx, dy), textcoords="offset points",
        ha=ha, va="center",
        fontsize=MICRO_FS, fontweight="bold", color=INK,
        bbox=dict(boxstyle="round,pad=0.30", fc="white",
                  ec=ACCENT, lw=1.0, alpha=0.98),
        arrowprops=dict(arrowstyle="-", color=ACCENT, lw=1.0),
        zorder=6,
    )

ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)


# -- Scale bar in metres (we are in a projected CRS) --------------------------
def draw_scale_bar(ax, length_m=500, label=None, pad_frac=0.035):
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    xr = xlim[1] - xlim[0]; yr = ylim[1] - ylim[0]
    x0 = xlim[0] + pad_frac * xr
    y0 = ylim[0] + pad_frac * yr
    x1 = x0 + length_m
    ax.plot([x0, x1], [y0, y0], color=INK, lw=4.0, solid_capstyle="butt", zorder=7)
    ax.plot([x0, (x0 + x1) / 2], [y0, y0],
            color="white", lw=2.0, solid_capstyle="butt", zorder=8)
    tick_dy = 0.008 * yr
    for xv, lab in [(x0, "0"), (x1, f"{length_m:,} m")]:
        ax.plot([xv, xv], [y0 - tick_dy, y0 + tick_dy], color=INK, lw=1.0, zorder=8)
        ax.text(xv, y0 + 2.8 * tick_dy, lab,
                ha="center", va="bottom", fontsize=MICRO_FS, color=INK_SOFT)
    if label:
        ax.text((x0 + x1) / 2, y0 - 3.2 * tick_dy, label,
                ha="center", va="top", fontsize=MICRO_FS, color=INK_SOFT)


def draw_north_arrow(ax, pad_frac=0.04, size_frac=0.055):
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    xr = xlim[1] - xlim[0]; yr = ylim[1] - ylim[0]
    size = size_frac * yr
    cx = xlim[1] - pad_frac * xr
    cy = ylim[1] - (pad_frac + size_frac) * yr
    ax.add_patch(FancyArrowPatch(
        (cx, cy), (cx, cy + size),
        arrowstyle="-|>", mutation_scale=18,
        color=INK, lw=1.8, zorder=7,
    ))
    ax.text(cx, cy + size + 0.008 * yr, "N",
            ha="center", va="bottom",
            fontsize=MICRO_FS + 1.5, fontweight="bold", color=INK)


draw_scale_bar(ax, length_m=500)
draw_north_arrow(ax)


# Data provenance footer.
ax.text(
    0.5, -0.04,
    "Data: OpenStreetMap via OSMnx · CRS: local UTM (metres) · "
    "Shortest path: length-weighted, between the graph's SW-most and NE-most nodes.",
    transform=ax.transAxes, ha="center", va="top",
    fontsize=MICRO_FS, color=INK_SOFT,
)

plt.show()


## 9. Exercises

Three prompts, each one harder than the last. Start with whichever matches your current comfort level.

### 9a. One deliberate figure (warm-up, repeating the core lesson)

Build one deliberate street-network figure of your own. The goal is not simply to plot the graph, but to *justify the representation*.

A strong submission should include:

1. one weak comparison figure that shows what goes wrong when spatial fidelity is ignored;
2. one final figure in geographic or projected space;
3. a short note explaining why the final representation fits the analytical question better;
4. at least one explicit design decision about whether nodes should be shown, hidden, or de-emphasized.

### 9b. Full four-stage walkthrough in a city of your choice

Pick a walkable district (your city, your home town, a neighbourhood you have lived in), and reproduce the four-stage progression from §3–§5:

1. spring layout (as a weak baseline);
2. geographic node coordinates;
3. full street geometry from GeoDataFrames;
4. projected local-metric CRS.

Write one sentence under each figure explaining what the representation preserves and what it loses.

### 9c. Name the main streets

Repeat the edge-betweenness analysis from §6d for the district you picked in 9b. Identify the **top 5 streets by mean edge-betweenness** — use the `name` attribute stored on each OSM edge. Do the results match your mental map of the district? If not, what does that tell you about *topological* importance versus *perceived* importance?

Hint:

```python
edges_bc_gdf["street_name"] = [
    (d.get("name") if isinstance(d.get("name"), str) else
     (d.get("name") or [None])[0])
    for d in (C_proj.get_edge_data(u, v, key) for u, v, key in
              zip(edges_bc_gdf["u"], edges_bc_gdf["v"], edges_bc_gdf["key"]))
]
(
    edges_bc_gdf.dropna(subset=["street_name"])
    .groupby("street_name")["edge_bc"].mean()
    .sort_values(ascending=False)
    .head(5)
)
```

In [ ]:
# Exercise scaffold (fill in)
#
# 1. Load a place with OSMnx:
#        H = ox.graph_from_place("Turin, Italy", network_type="walk")
#    or a bounded point:
#        H = ox.graph_from_point((45.0703, 7.6869), dist=1500, network_type="walk")
#
# 2. Build one deliberately weak non-spatial view (spring_layout).
# 3. Build one faithful geographic or projected view (graph_to_gdfs + project_graph).
# 4. Add a short markdown interpretation explaining what the faithful figure preserves.
# 5. (Optional, for 9c) Compute edge betweenness and annotate the top 5 streets by name.
